# XFAL ETL — Local PostgreSQL

Normalize `Data/Data.csv` into Company, Department, and Employee tables, then load into a local Postgres instance.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
CSV_PATH = ROOT / "Data" / "Data.csv"

df = pd.read_csv(CSV_PATH)

df["company_id"] = (
    pd.factorize(
        df["company_name"].astype(str)
        + df["address"].astype(str)
        + df["city"].astype(str)
        + df["state"].astype(str)
        + df["zip"].astype(str)
    )[0]
    + 1
)
df["department_id"] = pd.factorize(df["department"].astype(str))[0] + 1

df_company = df[["company_id", "company_name", "address", "city", "state", "zip"]].drop_duplicates()
df_department = (
    df[["department_id", "department"]]
    .drop_duplicates()
    .rename(columns={"department": "department_name"})
)
df_employee = df[["first_name", "last_name", "phone1", "phone2", "email", "company_id", "department_id"]]

print(df_company.shape, df_department.shape, df_employee.shape)
df_company.head()

In [ ]:
import os
import psycopg2
from psycopg2 import extras

connection = psycopg2.connect(
    dbname=os.getenv("POSTGRES_DB", "xfal"),
    user=os.getenv("POSTGRES_USER", "postgres"),
    password=os.getenv("POSTGRES_PASSWORD", "example"),
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=os.getenv("POSTGRES_PORT", "5432"),
)
print("Connected")

In [ ]:
def insert_into_table(connection, df, table_name):
    tuples = [tuple(x) for x in df.to_numpy()]
    cols = ",".join(list(df.columns))
    query = f"INSERT INTO {table_name} ({cols}) VALUES %s ON CONFLICT DO NOTHING"
    cursor = connection.cursor()
    try:
        extras.execute_values(cursor, query, tuples, template=None)
        connection.commit()
        print(f"Inserted into {table_name}: {len(tuples)} rows")
    except psycopg2.Error as e:
        connection.rollback()
        print(f"Error inserting into {table_name}: {e}")
    finally:
        cursor.close()

insert_into_table(connection, df_company, "company")
insert_into_table(connection, df_department, "department")
insert_into_table(connection, df_employee, "employee")
connection.close()